# Análise Comparativa de Modelos para Sumarização de Textos Científicos

Este notebook apresenta a comparação entre os modelos GPT-2, DistilGPT-2 e BERTimbau
na tarefa de sumarização automática de textos científicos em português.

In [ ]:
import sys
import os
import json
import time

sys.path.insert(0, os.path.abspath(".."))

from summarization import (
    ModelLoader,
    Summarizer,
    ExtractiveSummarizer,
    Seq2SeqSummarizer,
    TextPreprocessor,
)
from summarization.model_loader import SUPPORTED_MODELS
from evaluation import SummaryEvaluator, SemanticEvaluator
from data.corpus import Corpus
from utils import Timer, set_seed

import matplotlib.pyplot as plt
import numpy as np

# reprodutibilidade: mesma seed do experimento via CLI
set_seed(42)

## 1. Carregamento do Corpus

Utilizamos um corpus de 10 textos científicos em português com resumos de referência.

In [ ]:
corpus = Corpus()
samples = corpus.get_builtin_samples()
texts, references = corpus.get_texts_and_references()

print(f"Total de amostras: {len(samples)}\n")
for s in samples:
    print(f"  - {s['id']}: {s['title']}")

## 2. Configuracao dos Modelos

Modelos avaliados:
- **GPT-2** (`gpt2`): generativo, pre-treinado em ingles
- **DistilGPT-2** (`distilgpt2`): generativo, versao destilada
- **BERTimbau** (`neuralmind/bert-base-portuguese-cased`): extrativo, por similaridade de embeddings
- **PTT5-summ** (`recogna-nlp/ptt5-base-summ`): abstrativo, ajustado para sumarizacao em portugues

Decodificacao deterministica (greedy nos causais, beam no seq2seq) com `set_seed(42)`.

In [ ]:
MODELS = ["gpt2", "distilgpt2", "bertimbau", "ptt5-summ"]
NUM_SAMPLES = len(texts)  # corpus completo

test_texts = texts[:NUM_SAMPLES]
test_references = references[:NUM_SAMPLES]

print(f"Modelos: {MODELS}")
print(f"Amostras para teste: {NUM_SAMPLES}")

## 3. Geração de Resumos

Executamos cada modelo sobre as amostras de teste e medimos o tempo de execução.

In [ ]:
def resumir(model_name, textos):
    """Despacha para o sumarizador certo, com decodificacao deterministica."""
    tipo = SUPPORTED_MODELS[model_name]["type"]
    if tipo == "extractive":
        s = ExtractiveSummarizer()
        return [s.generate_summary(t, num_sentences=3) for t in textos]
    if tipo == "seq2seq":
        s = Seq2SeqSummarizer(model_name=model_name)
        return [s.generate_summary(t) for t in textos]  # beam deterministico
    s = Summarizer(model_name=model_name)
    return [s.generate_summary(t, do_sample=False) for t in textos]  # greedy


results = {}
for model_name in MODELS:
    print(f"\n{'='*60}")
    print(f"Processando: {model_name}")
    print(f"{'='*60}")
    with Timer() as t:
        try:
            summaries = resumir(model_name, test_texts)
        except Exception as e:
            print(f"Erro ao processar {model_name}: {e}")
            continue
    results[model_name] = {
        "summaries": summaries,
        "time": t.elapsed,
        "time_str": t.elapsed_str,
    }
    print(f"Tempo total: {t.elapsed_str}  |  resumos: {len(summaries)}")

## 4. Exemplos de Resumos Gerados

Comparação lado a lado dos resumos produzidos por cada modelo.

In [ ]:
for i in range(min(3, NUM_SAMPLES)):
    print(f"\n{'='*70}")
    print(f"AMOSTRA {i+1}: {samples[i]['title']}")
    print(f"{'='*70}")
    print(f"\nResumo de referencia:")
    print(f"  {test_references[i]}")

    for model_name in MODELS:
        if model_name in results and i < len(results[model_name]["summaries"]):
            print(f"\nResumo [{model_name}]:")
            print(f"  {results[model_name]['summaries'][i]}")

## 5. Avaliação com ROUGE

Métricas ROUGE medem a sobreposição de n-gramas entre o resumo gerado e o resumo de referência:
- **ROUGE-1**: unigramas
- **ROUGE-2**: bigramas
- **ROUGE-L**: maior subsequência comum

In [ ]:
rouge_eval = SummaryEvaluator()
semantic_eval = SemanticEvaluator()
rouge_results = {}
semantic_results = {}

for model_name in MODELS:
    if model_name not in results or not results[model_name]["summaries"]:
        continue
    hyps = results[model_name]["summaries"]
    # ordem correta: evaluate_batch(referencias, hipoteses)
    rouge_results[model_name] = rouge_eval.evaluate_batch(test_references, hyps)
    semantic_results[model_name] = semantic_eval.evaluate_batch(test_references, hyps)

    print(f"\n--- {model_name.upper()} ---")
    print(rouge_eval.format_results(rouge_results[model_name]))

## 6. Tabela Comparativa

In [ ]:
print(f"{'Modelo':<15} {'ROUGE-1 F':>12} {'ROUGE-2 F':>12} {'ROUGE-L F':>12} {'Sem.':>8} {'Tempo':>10}")
print("-" * 73)

for model_name in MODELS:
    if model_name not in rouge_results:
        continue
    r = rouge_results[model_name]
    sem = semantic_results[model_name]["semantic_similarity_mean"]
    print(
        f"{model_name:<15} "
        f"{r['rouge1']['fmeasure']:>12.4f} "
        f"{r['rouge2']['fmeasure']:>12.4f} "
        f"{r['rougeL']['fmeasure']:>12.4f} "
        f"{sem:>8.4f} "
        f"{results[model_name]['time_str']:>10}"
    )

## 7. Visualização dos Resultados

In [ ]:
# Paleta Okabe-Ito (segura para daltonismo)
model_names = [m for m in MODELS if m in rouge_results]
rouge1_scores = [rouge_results[m]["rouge1"]["fmeasure"] for m in model_names]
rouge2_scores = [rouge_results[m]["rouge2"]["fmeasure"] for m in model_names]
rougeL_scores = [rouge_results[m]["rougeL"]["fmeasure"] for m in model_names]

x = np.arange(len(model_names))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width, rouge1_scores, width, label="ROUGE-1", color="#0072B2")
bars2 = ax.bar(x, rouge2_scores, width, label="ROUGE-2", color="#E69F00")
bars3 = ax.bar(x + width, rougeL_scores, width, label="ROUGE-L", color="#009E73")

ax.set_xlabel("Modelo")
ax.set_ylabel("F-measure")
ax.set_title("Comparacao ROUGE entre Modelos")
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.legend(frameon=False)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f"{height:.3f}", xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=8)

os.makedirs("../experiments/results", exist_ok=True)
plt.tight_layout()
plt.savefig("../experiments/results/nb_rouge.png", dpi=150)
plt.show()

## 8. Tempo de Execução

In [ ]:
times = [results[m]["time"] for m in model_names]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(model_names, times, color="#0072B2")

ax.set_xlabel("Modelo")
ax.set_ylabel("Tempo (segundos)")
ax.set_title("Tempo de Execucao por Modelo")

for bar in bars:
    height = bar.get_height()
    ax.annotate(f"{height:.1f}s", xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points", ha="center", va="bottom")

os.makedirs("../experiments/results", exist_ok=True)
plt.tight_layout()
plt.savefig("../experiments/results/nb_tempo.png", dpi=150)
plt.show()

## 9. Salvando Resultados

In [ ]:
export_data = {}
for model_name in model_names:
    export_data[model_name] = {
        "rouge_scores": rouge_results[model_name],
        "semantic_scores": semantic_results[model_name],
        "execution_time": results[model_name]["time"],
        "num_samples": len(results[model_name]["summaries"]),
        "summaries": results[model_name]["summaries"],
    }

os.makedirs("../experiments/results", exist_ok=True)
with open("../experiments/results/resultados_notebook.json", "w", encoding="utf-8") as f:
    json.dump(export_data, f, ensure_ascii=False, indent=2)

print("Resultados salvos em experiments/results/resultados_notebook.json")

## 10. Conclusões Preliminares

A análise acima permite identificar:

1. Qual modelo obteve melhores scores ROUGE (qualidade do resumo)
2. Trade-off entre qualidade e velocidade de cada modelo
3. Diferenças entre abordagem extrativa (BERTimbau) e generativa (GPT-2/DistilGPT-2)

**Próximos passos:**
- Adicionar LLaMA-2 à comparação quando acesso for aprovado
- Avaliar similaridade semântica com BERTScore
- Aumentar corpus de avaliação com mais textos científicos
- Testar diferentes parâmetros de geração (temperatura, top_p)